# Creating the Prediction pipeline

## Module Loading

In [1]:
from google.colab import drive
from shutil import copy2
from duckdb import connect as dcon
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats
import plotly.express as px
from pylab import rcParams
import pandas as pd
import numpy as np
from warnings import filterwarnings

%matplotlib inline
darkmodel = True
rcParams['figure.figsize'] = (12,6)
pd.options.display.float_format = '{:,.2f}'.format
filterwarnings('ignore', category=FutureWarning)

In [2]:
if darkmodel:
    # 1. Define a sophisticated E-commerce color palette
    # These colors are chosen for high contrast against the #212946 background
    colors = [
        "#08F7FE",  # Cyan Glow
        "#FE53BB",  # Neon Pink
        "#F5D300",  # Cyber Yellow
        "#00ff41",  # Matrix Green
        "#9467bd",  # Royal Purple
    ]

    # 2. Enhanced Dictionary with Complex Styling
    refined_dark_style = {
        # Background and Canvas
        "figure.facecolor": "#212946",
        "axes.facecolor": "#212946",
        "savefig.facecolor": "#212946",

        # Grid Sophistication
        "axes.grid": True,
        "axes.grid.which": "both",
        "grid.color": "#2A3459",
        "grid.linewidth": "1",
        "grid.alpha": 0.5,

        # Typography & Labels (Optimized for readability)
        "text.color": "#E2E2E2",
        "axes.labelcolor": "#E2E2E2",
        "axes.labelsize": 14,
        "axes.titlesize": 18,
        "axes.titleweight": "bold",
        "axes.titlepad": 20,
        "xtick.color": "#8E9CC3",
        "ytick.color": "#8E9CC3",
        "font.size": 12,

        # Spines (Clean aesthetic)
        "axes.spines.left": False,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "axes.spines.bottom": True,
        "axes.edgecolor": "#2A3459",

        # Line & Marker Settings
        "lines.linewidth": 2.5,
        "lines.markersize": 8,
        "axes.prop_cycle": plt.cycler(color=colors),
    }

    plt.rcParams.update(refined_dark_style)

else:
    # 1. Defining the "Paper & Ink" Palette
    # Deep Blue #003366 | Oxide Red #A52A2A
    ecom_vintage_colors = [
        "#003366",  # Oxford Blue (Primary)
        "#A52A2A",  # Oxide Red (Comparison)
        "#006400",  # Dark Green (Success Metrics)
        "#704214",  # Sepia (Neutral)
    ]

    vintage_style = {
        # Background - The specific parchment hex you requested
        "figure.facecolor": "#f7e4b7",
        "axes.facecolor": "#f7e4b7",
        "savefig.facecolor": "#f7e4b7",

        # Grid - Subtle contrast using a darker version of the background
        "axes.grid": True,
        "grid.color": "#e2d1a8",
        "grid.linestyle": "-",
        "grid.linewidth": 1.0,

        # Typography - Deep Charcoal/Blue instead of pure black for a softer feel
        "text.color": "#2C2C2C",
        "axes.labelcolor": "#2C2C2C",
        "xtick.color": "#5D5D5D",
        "ytick.color": "#5D5D5D",
        "axes.titlesize": 16,
        "axes.titleweight": "bold",
        "axes.titlepad": 15,
        "font.size": 11,

        # Spines - Classic 'L-frame' for publication
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.spines.left": True,
        "axes.spines.bottom": True,
        "axes.edgecolor": "#2C2C2C",
        "axes.linewidth": 1.5,

        # Data Point Styling
        "axes.prop_cycle": plt.cycler(color=ecom_vintage_colors),
        "lines.linewidth": 2.2,
        "lines.markersize": 8,
        "patch.edgecolor": "#f7e4b7", # Borders on bars/pie slices
    }

    plt.rcParams.update(vintage_style)

## Data Test Loading

In [3]:
drive.mount('/content/drive')
pd.set_option('display.max_rows', 50)

def ListFiles(Dirs):
    errormsg = f"Error: Directory '{Dirs}' does not exist or is not a directory."
    assert os.path.isdir(Dirs), errormsg
    file_data = list()
    for item in os.listdir(Dirs):
        item_path = os.path.join(Dirs, item)
        if os.path.isfile(item_path):
            try:
                size_bytes = os.path.getsize(item_path)
                size_mb = size_bytes / (1024 * 1024)  # Convert bytes to MB
                file_data.append({'File Name': item, 'Size (MB)': size_mb})
            except Exception as err:
                print(f"Could not get size for {item_path}: {err}")

    Files = pd.DataFrame(file_data)
    return Files

Mounted at /content/drive


In [4]:
MyFiles = ListFiles('/content/drive/MyDrive/Colab Notebooks')
DatFilename = MyFiles[~MyFiles['File Name'].str.contains('.ipynb', na = False)]
display(DatFilename)

,File Name,Size (MB)
76,MasterData.parquet,145.58
79,InstaCart.db,263.76
82,xgboost_ltr_model.json,2.43
83,catboost_ltr_model.cbm,0.32
84,lightgbm_ltr_model.txt,6.21
85,lgbm_optuna_ranker_model.txt,0.58
86,test_df.parquet,3.84


In [5]:
test_df = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/test_df.parquet')
display(test_df.head())

,order_id,user_id,product_id,aisle_id,department_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,add_to_cart_order,reordered,user_total_orders,user_avg_days_between,user_avg_cart_pos,user_total_reorders,index,prod_total_reorders,prod_reorder_rate,prod_order_count,prod_avg_cart_pos
0,252513,110001,41588,14,20,38,4,19,3.00,8,1,38,3.00,6.50,9,32736,256,0.65,391,8.82
1,252513,110001,49628,120,16,38,4,19,3.00,12,1,38,3.00,6.50,9,39078,123,0.66,186,8.49
2,252654,42214,23178,98,7,10,5,16,12.00,3,0,10,12.00,2.00,1,18299,108,0.61,176,7.26
3,252654,42214,24838,91,16,10,5,16,12.00,1,0,10,12.00,2.00,1,19563,1615,0.74,2169,6.28
4,252654,42214,46667,83,4,10,5,16,12.00,2,1,10,12.00,2.00,1,36739,1241,0.62,2007,9.49


In [6]:
db_path = '/content/drive/MyDrive/Colab Notebooks/InstaCart.db'
con = dcon(database=db_path, read_only=True)
tables = con.execute("PRAGMA show_tables;").fetchdf()
print("Tables in InstaCart.db:")
display(tables)

Tables in InstaCart.db:


,name
0,AISLE
1,DepartmentData
2,FullTrainData
3,OrderTest
4,OrderTrain
5,OrdersDetails
6,ProductsData


In [7]:
for table_name in tables['name']:
    print(f"\n--- Sample from Table: {table_name} ---")
    query = f"SELECT * FROM {table_name} LIMIT 3;"
    df_sample = con.execute(query).fetchdf()
    display(df_sample)


--- Sample from Table: AISLE ---


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars



--- Sample from Table: DepartmentData ---


,department_id,department
0,1,frozen
1,2,other
2,3,bakery



--- Sample from Table: FullTrainData ---


...

,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0



--- Sample from Table: OrdersDetails ---


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,08,NaN
1,2398795,1,prior,2,3,07,15.00
2,473747,1,prior,3,3,12,21.00



--- Sample from Table: ProductsData ---


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7


In [8]:
Q3 = f"SELECT * FROM ProductsData;"
DataProduct = con.execute(Q3).fetchdf()

## Forming Predict Function

### Predicted only which item been ever bought

In [9]:
import xgboost as xgb
import numpy as np

def load_xgb(model_path: str):
    assert os.path.exists(model_path), f"Model file '{model_path}' does not exist."
    model = xgb.Booster()
    model.load_model(model_path)
    return model


In [10]:
import xgboost as xgb
import pandas as pd
from typing import List, Dict
from joblib import Parallel, delayed

def process_single_user(user_id: int,
                        full_dataset: pd.DataFrame,
                        bst: xgb.Booster,
                        features: List[str],
                       ):
    """Helper function to process one user at a time for parallel workers."""
    # 1. Filter data for this specific user
    user_df = full_dataset[full_dataset['user_id'] == user_id].copy()

    if user_df.empty:
        return user_id, None

    # 2. Predict
    dmatrix = xgb.DMatrix(user_df[features])
    user_df['score'] = bst.predict(dmatrix)

    # 3. Format result: list of tuples (product_id, score)
    top_items = (
        user_df.sort_values(by='score', ascending=False)
        .head(10)[['product_id', 'score']]
        .to_records(index=False)
        .tolist()
    )
    return user_id, top_items

def recommend_for_users_parallel(user_ids: List[int],
                                 full_dataset: pd.DataFrame,
                                 model_path: str,
                                 features: List[str],
                                 n_jobs: int = -1):
    # 1. Load model ONCE in the main process
    bst = load_xgb(model_path)
    # Ensure XGBoost uses 1 thread per worker to avoid CPU thrashing
    bst.set_param('nthread', 1)

    # 2. Execute in Parallel
    # n_jobs=-1 uses all available CPU cores
    results = Parallel(n_jobs=n_jobs)(
        delayed(process_single_user)(uid, full_dataset, bst, features)
        for uid in user_ids
    )

    # 3. Convert list of tuples to Dictionary {user_id: [rec_list]}
    recommendation_dict = {uid: recs for uid, recs in results if recs is not None}
    return recommendation_dict

In [19]:
import pandas as pd
import numpy as np

def get_readable_recommendations(rec_dict, data_product_df):
    """
    Converts the recommendation dictionary into a formatted DataFrame
    with Product Names, Magnitudes, and Remark column.
    """
    # 1. Flatten the dictionary into a list of rows
    # rec_dict format: {user_id: [(prod_id, score), ...]}
    rows = list()
    for user_id, recs in rec_dict.items():
        for prod_id, score in recs:
            rows.append({
                "userID": user_id,
                "productID": prod_id,
                "Magnitude": score
            })

    # 2. Create a temporary DataFrame from the results
    df_results = pd.DataFrame(rows)

    # 3. Merge with DataProduct to get names
    final_df = df_results.merge(
        data_product_df[['product_id', 'product_name']],
        left_on='productID',
        right_on='product_id',
        how='left'
    )

    # 4. Cleanup and Rename columns
    final_df = final_df.rename(columns={'product_name': 'productName'})

    # 5. Add Remark Column
    conditions = [
        final_df["Magnitude"] >= 0.2,
        (final_df["Magnitude"] > 0.0) & (final_df["Magnitude"] < 0.2),
        final_df["Magnitude"] <= 0.0
    ]

    choices = [
        "intention to buy again",
        "not really necessary",
        "I hate it!"
    ]

    final_df["Remark"] = np.select(conditions, choices, default="not really necessary")

    # 6. Reorder columns
    final_df = final_df[
        ["userID", "productID", "productName", "Magnitude", "Remark"]
    ]

    # 7. Sort
    final_df = final_df.sort_values(
        by=['userID', 'Magnitude'],
        ascending=[True, False]
    )
    return final_df

In [20]:
from pathlib import Path

dir_path = Path("/content/drive/MyDrive/Colab Notebooks")
model_filename = 'xgboost_ltr_model.json'
MFP = dir_path / model_filename

In [21]:
# the usage of Features
TheFeature = ['order_number', 'order_dow', 'days_since_prior_order',
       'add_to_cart_order', 'user_total_orders', 'user_avg_days_between',
       'user_avg_cart_pos', 'user_total_reorders', 'prod_total_reorders',
       'prod_reorder_rate', 'prod_order_count', 'prod_avg_cart_pos']

In [22]:
user_list = [110001, 42214]
final_recs = recommend_for_users_parallel(user_list, test_df, MFP, TheFeature)

In [23]:
print(final_recs)

{110001: [(21137, 2.193127393722534), (26209, 1.5212301015853882), (2781, 1.0138267278671265), (21616, 0.7028533816337585), (33731, 0.43611109256744385), (36070, 0.001868915162049234), (27156, -0.04143115133047104), (41588, -0.3019513189792633), (13517, -0.47096434235572815), (49628, -0.8822891116142273)], 42214: [(24838, 1.2657248973846436), (46667, -0.07607019692659378), (23178, -0.3764210045337677)]}


In [24]:
final_report = get_readable_recommendations(final_recs, DataProduct)
display(final_report)

,userID,productID,productName,Magnitude,Remark
10,42214,24838,Unsweetened Almondmilk,1.27,intention to buy again
11,42214,46667,Organic Ginger Root,-0.08,I hate it!
12,42214,23178,Pure Lemon Juice,-0.38,I hate it!
0,110001,21137,Organic Strawberries,2.19,intention to buy again
1,110001,26209,Limes,1.52,intention to buy again
2,110001,2781,Chipotle Lime Meat-Free Crispy Fingers,1.01,intention to buy again
3,110001,21616,Organic Baby Arugula,0.70,intention to buy again
4,110001,33731,Grated Parmesan,0.44,intention to buy again
5,110001,36070,"Super Spinach! Baby Spinach, Baby Bok Choy, Sw...",0.00,not really necessary
6,110001,27156,Organic Black Beans,-0.04,I hate it!


### Get item Rank with same AisleyID and DepartmentID to try to recommended

In [ ]:
def get_user_candidates(user_id: int, full_dataset: pd.DataFrame, data_product: pd.DataFrame, features: list):
    # 1. Get user history
    user_history = full_dataset[full_dataset['user_id'] == user_id]
    if user_history.empty:
        return None

    # 2. Get user's preferred categories
    user_aisles = user_history['aisle_id'].unique()
    user_depts = user_history['department_id'].unique()
    bought_ids = user_history['product_id'].unique()

    # 3. Filter DataProduct for new items in those categories
    # This ensures we are only looking at the product catalog
    new_items = data_product[
        ((data_product['aisle_id'].isin(user_aisles)) |
         (data_product['department_id'].isin(user_depts))) &
        (~data_product['product_id'].isin(bought_ids))
    ].copy()

    if new_items.empty:
        return None

    # 4. Synthesis: Attach user-level features to the new product rows
    user_stats = user_history.iloc[0]
    for col in features:
        # If the feature isn't in the product table, take it from user history
        if col not in new_items.columns:
            new_items[col] = user_stats[col]

    # Explicitly set reordered to 0 for these discovery items
    if 'reordered' in features:
        new_items['reordered'] = 0

    # Ensure user_id exists for tracking
    new_items['user_id'] = user_id

    return new_items

In [ ]:
def process_single_user_expanded(user_id: int,
                                 full_dataset: pd.DataFrame,
                                 data_product: pd.DataFrame,
                                 bst: xgb.Booster,
                                 features: list):
    # Get the candidates
    predict_df = get_user_candidates(user_id, full_dataset, data_product, features)

    # FIX: Check if None first to avoid 'tuple' or 'NoneType' errors
    if predict_df is None:
        return user_id, None

    if predict_df.empty:
        return user_id, None

    # Prediction
    # We slice only the features the model was trained on
    dmat = xgb.DMatrix(predict_df[features])
    predict_df['score'] = bst.predict(dmat)

    # Filter: Positive Scores only
    top_df = predict_df[predict_df['score'] > 0].copy()

    if top_df.empty:
        return user_id, None

    # Sort and take Top 10
    top_results = (
        top_df.sort_values(by='score', ascending=False)
        .head(10)[['product_id', 'score']]
    )

    # Return as list of records
    return user_id, list(top_results.to_records(index=False))

In [ ]:
def run_recommendation_pipeline(user_ids, full_dataset, data_product, model_path, features, n_jobs=-1):
    bst = load_xgb(model_path)
    bst.set_param('nthread', 1)

    results = Parallel(n_jobs=n_jobs)(
        delayed(process_single_user_expanded)(uid, full_dataset, data_product, bst, features)
        for uid in user_ids
    )

    return {uid: recs for uid, recs in results if recs is not None}

In [ ]:
TheFeature = ['order_number', 'order_dow', 'days_since_prior_order',
       'add_to_cart_order', 'user_total_orders', 'user_avg_days_between',
       'user_avg_cart_pos', 'user_total_reorders', 'prod_total_reorders',
       'prod_reorder_rate', 'prod_order_count', 'prod_avg_cart_pos']

user_list = [110001, 42214]

# 3. Execute the pipeline
if __name__ == "__main__":
    # This calls the 3-function system we built
    recommendation_dict = run_recommendation_pipeline(
        user_ids=user_list,
        full_dataset=test_df,      # Your historical/test data
        data_product=DataProduct,  # Your product catalog
        model_path=MFP,
        features=TheFeature,
        n_jobs=-1                  # Use all CPU cores
    )

    # 4. Convert the dictionary to your requested DataFrame format
    # Using the helper function we created earlier
    final_report_02 = get_readable_recommendations(recommendation_dict, DataProduct)


KeyError: 'productID'

In [ ]:
models = load_xgb(MFP)
models.set_param('nthread', 1)

x = process_single_user_expanded(user_id = 110001,
                                 full_dataset = test_df,
                                 data_product = DataProduct,
                                 bst = models,
                                 features = TheFeature)

In [25]:
usid = 42214
user_history = test_df[test_df['user_id'] == usid]

# 2. Get user's preferred categories
user_aisles = user_history['aisle_id'].unique()
user_depts = user_history['department_id'].unique()
bought_ids = user_history['product_id'].unique()

print(bought_ids)

# 3. Filter DataProduct for new items in those categories
# This ensures we are only looking at the product catalog
new_items = DataProduct[
    ((DataProduct['aisle_id'].isin(user_aisles)) &
        (DataProduct['department_id'].isin(user_depts))) &
    (~DataProduct['product_id'].isin(bought_ids))
].copy()

display(new_items.shape)

print(user_history.iloc[0])

display(user_history)

[23178 24838 46667]


(1651, 4)

order_id                  252654
user_id                    42214
product_id                 23178
aisle_id                      98
department_id                  7
order_number                  10
order_dow                      5
order_hour_of_day             16
days_since_prior_order     12.00
add_to_cart_order              3
reordered                      0
user_total_orders             10
user_avg_days_between      12.00
user_avg_cart_pos           2.00
user_total_reorders            1
index                      18299
prod_total_reorders          108
prod_reorder_rate           0.61
prod_order_count             176
prod_avg_cart_pos           7.26
Name: 2, dtype: object


,order_id,user_id,product_id,aisle_id,department_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,add_to_cart_order,reordered,user_total_orders,user_avg_days_between,user_avg_cart_pos,user_total_reorders,index,prod_total_reorders,prod_reorder_rate,prod_order_count,prod_avg_cart_pos
2,252654,42214,23178,98,7,10,5,16,12.00,3,0,10,12.00,2.00,1,18299,108,0.61,176,7.26
3,252654,42214,24838,91,16,10,5,16,12.00,1,0,10,12.00,2.00,1,19563,1615,0.74,2169,6.28
4,252654,42214,46667,83,4,10,5,16,12.00,2,1,10,12.00,2.00,1,36739,1241,0.62,2007,9.49


In [ ]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 276986 entries, 0 to 276985
Data columns (total 20 columns):
 #   Column                  Non-Null Count   Dtype   
---  ------                  --------------   -----   
 0   order_id                276986 non-null  int64   
 1   user_id                 276986 non-null  int64   
 2   product_id              276986 non-null  int64   
 3   aisle_id                276986 non-null  int64   
 4   department_id           276986 non-null  int64   
 5   order_number            276986 non-null  int64   
 6   order_dow               276986 non-null  int64   
 7   order_hour_of_day       276986 non-null  category
 8   days_since_prior_order  276986 non-null  float64 
 9   add_to_cart_order       276986 non-null  int64   
 10  reordered               276986 non-null  int64   
 11  user_total_orders       276986 non-null  int64   
 12  user_avg_days_between   276986 non-null  float64 
 13  user_avg_cart_pos       276986 non-null  float64 
 14  user

In [ ]:
display(final_report_02)

In [ ]:
def get_readable_recommendations(rec_dict, data_product_df):
    """
    Converts the recommendation dictionary into a formatted DataFrame.
    """
    if not rec_dict:
        print("No recommendations found with positive scores.")
        return pd.DataFrame(columns=["userID", "productID", "productName", "Magnitude"])

    rows = []
    for user_id, recs in rec_dict.items():
        for item in recs:
            # item is a tuple/record like (product_id, score)
            rows.append({
                "userID": user_id,
                "productID": item[0],  # Explicitly take first element
                "Magnitude": item[1]   # Explicitly take second element
            })

    df_results = pd.DataFrame(rows)

    # Merge with DataProduct
    final_df = df_results.merge(
        data_product_df[['product_id', 'product_name']],
        left_on='productID',
        right_on='product_id',
        how='left'
    )

    # Cleanup
    final_df = final_df.rename(columns={'product_name': 'productName'})
    return final_df[["userID", "productID", "productName", "Magnitude"]]

In [ ]:
def process_single_user_expanded(user_id, full_dataset, data_product, bst, features):
    # 1. Get candidates
    predict_df = get_user_candidates(user_id, full_dataset, data_product, features)

    if predict_df is None or predict_df.empty:
        return user_id, None

    # 2. Predict
    dmat = xgb.DMatrix(predict_df[features])
    predict_df['score'] = bst.predict(dmat)

    # 3. Filter and Sort
    # Only Magnitude > 0
    top_df = predict_df[predict_df['score'] > 0].sort_values(by='score', ascending=False).head(10)

    if top_df.empty:
        return user_id, None

    # Return as a simple list of lists [ [prod_id, score], ... ]
    return user_id, top_df[['product_id', 'score']].values.tolist()

In [ ]:
def get_user_candidates(user_id, full_dataset, data_product, features):
    user_history = full_dataset[full_dataset['user_id'] == user_id]
    if user_history.empty:
        return None

    user_aisles = user_history['aisle_id'].unique()
    user_depts = user_history['department_id'].unique()
    bought_ids = user_history['product_id'].unique()

    # Find items in same categories user hasn't bought
    new_items = data_product[
        ((data_product['aisle_id'].isin(user_aisles)) |
         (data_product['department_id'].isin(user_depts))) &
        (~data_product['product_id'].isin(bought_ids))
    ].copy()

    if new_items.empty:
        return None

    # Sync features
    user_stats = user_history.iloc[0]
    for col in features:
        if col not in new_items.columns:
            new_items[col] = user_stats[col]

    if 'reordered' in features:
        new_items['reordered'] = 0

    return new_items

In [ ]:
res_id, res_list = process_single_user_expanded(110001, test_df, DataProduct, models, TheFeature)

# If it returns data, wrap it in a dict to test the formatter
test_dict = {res_id: res_list}
final_report = get_readable_recommendations(test_dict, DataProduct)
print(final_report)

TypeError: 'NoneType' object is not iterable

In [ ]:
test_df.head()

,order_id,user_id,product_id,aisle_id,department_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,add_to_cart_order,reordered,user_total_orders,user_avg_days_between,user_avg_cart_pos,user_total_reorders,index,prod_total_reorders,prod_reorder_rate,prod_order_count,prod_avg_cart_pos
0,252513,110001,41588,14,20,38,4,19,3.00,8,1,38,3.00,6.50,9,32736,256,0.65,391,8.82
1,252513,110001,49628,120,16,38,4,19,3.00,12,1,38,3.00,6.50,9,39078,123,0.66,186,8.49
2,252654,42214,23178,98,7,10,5,16,12.00,3,0,10,12.00,2.00,1,18299,108,0.61,176,7.26
3,252654,42214,24838,91,16,10,5,16,12.00,1,0,10,12.00,2.00,1,19563,1615,0.74,2169,6.28
4,252654,42214,46667,83,4,10,5,16,12.00,2,1,10,12.00,2.00,1,36739,1241,0.62,2007,9.49


In [ ]:
user_42214.to_markdown()

'|    |   order_id |   user_id |   product_id |   aisle_id |   department_id |   order_number |   order_dow |   order_hour_of_day |   days_since_prior_order |   add_to_cart_order |   reordered |   user_total_orders |   user_avg_days_between |   user_avg_cart_pos |   user_total_reorders |   index |   prod_total_reorders |   prod_reorder_rate |   prod_order_count |   prod_avg_cart_pos |\n|---:|-----------:|----------:|-------------:|-----------:|----------------:|---------------:|------------:|--------------------:|-------------------------:|--------------------:|------------:|--------------------:|------------------------:|--------------------:|----------------------:|--------:|----------------------:|--------------------:|-------------------:|--------------------:|\n|  2 |     252654 |     42214 |        23178 |         98 |               7 |             10 |           5 |                  16 |                       12 |                   3 |           0 |                  10 |        

In [ ]:
final_report[final_report['userID'] == 42214].to_markdown()

'|    |   userID |   productID | productName            |   Magnitude |\n|---:|---------:|------------:|:-----------------------|------------:|\n| 10 |    42214 |       24838 | Unsweetened Almondmilk |   1.26572   |\n| 11 |    42214 |       46667 | Organic Ginger Root    |  -0.0760702 |\n| 12 |    42214 |       23178 | Pure Lemon Juice       |  -0.376421  |'

In [ ]:
user_42214 = test_df[test_df['user_id'] == 42214]
display(user_42214)

,order_id,user_id,product_id,aisle_id,department_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,add_to_cart_order,reordered,user_total_orders,user_avg_days_between,user_avg_cart_pos,user_total_reorders,index,prod_total_reorders,prod_reorder_rate,prod_order_count,prod_avg_cart_pos
2,252654,42214,23178,98,7,10,5,16,12.00,3,0,10,12.00,2.00,1,18299,108,0.61,176,7.26
3,252654,42214,24838,91,16,10,5,16,12.00,1,0,10,12.00,2.00,1,19563,1615,0.74,2169,6.28
4,252654,42214,46667,83,4,10,5,16,12.00,2,1,10,12.00,2.00,1,36739,1241,0.62,2007,9.49


In [ ]:
user_110001 = test_df[test_df['user_id'] == 110001]
print(user_110001.shape[0], 'rows only.')
#display(user_110001)

12 rows only.


In [ ]:
user_42214 = test_df[test_df['user_id'] == 42214]
display(user_42214)DataProduct.sample(10).to_markdown()

SyntaxError: invalid syntax (2972593220.py, line 2)

In [ ]:
top_10 = recommend_for_user([110001, 42214], test_df, MFP, TheFeature)
print(top_10)

       product_id  prediction_score
41726       21137              2.19
41728       26209              1.52
3           24838              1.27
41723        2781              1.01
41727       21616              0.70
41730       33731              0.44
41731       36070              0.00
41729       27156             -0.04
4           46667             -0.08
0           41588             -0.30


In [ ]:
jt = test_df[test_df['user_id'].isin([110001, 42214])]
jt = jt[["order_id", "user_id", "product_id"]].sort_values(by="product_id")
display(jt)

,order_id,user_id,product_id
41722,252513,110001,2663
41723,252513,110001,2781
41724,252513,110001,3376
41725,252513,110001,13517
41726,252513,110001,21137
41727,252513,110001,21616
2,252654,42214,23178
3,252654,42214,24838
41728,252513,110001,26209
41729,252513,110001,27156


In [ ]:
model = load_xgb(MFP)
preds = predict_xgb(model, X_test)

In [ ]:
preds

array([ 1.0457449 ,  0.25461856, -1.1025273 , ...,  1.8161756 ,
       -0.82432204, -0.9713255 ], dtype=float32)

In [ ]:
import pandas as pd

TheFeature = ['order_number', 'order_dow', 'days_since_prior_order',
       'add_to_cart_order', 'user_total_orders', 'user_avg_days_between',
       'user_avg_cart_pos', 'user_total_reorders', 'prod_total_reorders',
       'prod_reorder_rate', 'prod_order_count', 'prod_avg_cart_pos']

TheFeatures = pd.Index(TheFeature)

In [ ]:
def Final_LTRpreps(Data : pd.DataFrame, Feature = list):
    df_sorted = Data.sort_values(['user_id']).reset_index(drop=True)
    X = df_sorted[Feature].values
    y = df_sorted['reordered'].values
    query_ids = df_sorted['user_id'].values
    # Hitung group (jumlah sampel per user)
    group = df_sorted.groupby('user_id').size().values
    return X, y, group, query_ids

In [ ]:
X_test, y_test, group_test, q_test = Final_LTRpreps(test_df, TheFeatures)

In [ ]:
test_df.sample(20).to_markdown()

'|        |   order_id |   user_id |   product_id |   aisle_id |   department_id |   order_number |   order_dow |   order_hour_of_day |   days_since_prior_order |   add_to_cart_order |   reordered |   user_total_orders |   user_avg_days_between |   user_avg_cart_pos |   user_total_reorders |   index |   prod_total_reorders |   prod_reorder_rate |   prod_order_count |   prod_avg_cart_pos |\n|-------:|-----------:|----------:|-------------:|-----------:|----------------:|---------------:|------------:|--------------------:|-------------------------:|--------------------:|------------:|--------------------:|------------------------:|--------------------:|----------------------:|--------:|----------------------:|--------------------:|-------------------:|--------------------:|\n| 222248 |    2885654 |    167409 |        24964 |         83 |               4 |              5 |           3 |                  12 |                       20 |                   6 |           1 |                  